---

# Veyra: Barn CCTV Cattle Detector

*Fine-tuning the CCTV detector that feeds Veyra's tracker, so that identities handed over in the scan zone are not lost whenever an animal is missed for a few frames.*

---

## Team Veyra

- Member 1 (Vision and data)
- Member 2
- Member 3
- Member 4

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Check**](#3)
4. [**Zero-Shot Baseline**](#4)
5. [**Fine-Tuning**](#5)
6. [**Evaluation**](#6)
7. [**Export**](#7)


---

# Introduction <a name="1"></a>

---

## Overview

*Veyra links a cow's identity, confirmed by a muzzle scan in a marked scan zone, to its track on a barn camera. A track only keeps that identity while the detector keeps seeing the animal: every missed detection can end the track, and a new track starts without an identity.*

*On ten minutes of real barn footage, the COCO-pretrained YOLOv8m detector found only about 40% of the calves, which left almost no identities intact after the hand-over. The tracker was not the bottleneck: ByteTrack and BoT-SORT produced nearly the same results.*

## Aim

*Fine-tune a single-class cattle detector on hand-labelled barn frames and raise detection recall on held-out days well above the zero-shot baseline, while keeping precision high, so that tracks stay continuous enough to carry identities.*

## Metric

***Recall at IoU 0.5** is the share of labelled calves that the detector finds. It is the primary metric here because a missed calf breaks its track and drops its identity. **Precision** guards against false detections, and **mAP@0.5** and **mAP@0.5:0.95** summarise the precision-recall trade-off.*

$$\text{Recall} = \frac{TP}{TP + FN} \qquad \text{Precision} = \frac{TP}{TP + FP}$$

## Dataset

*Data sourced from Hugging Face, 8-Calves (Fang et al., 2025, CC BY 4.0) at: https://huggingface.co/datasets/tonyFang04/8-calves*

*The split follows the authors' benchmark and is separated by recording day: 500 frames for training (morning feed on 2-3 March, afternoon feed on 1-3 March) and 100 frames for validation (morning feed on 4 March). Frames from the afternoon feed of 4 March are excluded, because the tracking test video comes from that session.*

*The dataset is used to adapt the detector to an angled barn camera, straw bedding, and crowded calves.*

```
eight_calves_yolo/
+-- data.yaml
+-- train/
|   +-- images/   (500 PNG frames, 800 x 600)
|   +-- labels/   (YOLO txt, class 0 = calf)
+-- val/
     +-- images/   (100 PNG frames)
     +-- labels/
```

## Approach: Fine-Tuned YOLOv8m

*YOLOv8m is a one-stage detector pretrained on COCO, which already contains a cow class. Fine-tuning keeps its general features and adapts the head to calves seen from a high, angled camera.*

*Training uses both T4 GPUs with distributed data parallelism, an 800-pixel input to keep small calves visible, and augmentation suited to an overhead view: flips in both directions, small rotations, and brightness changes.*

```
Barn frame [800 x 600]
    |
    +-- YOLOv8m backbone + neck (COCO pretrained)
         |
    Detection head -> calf boxes + confidence
         |
    Tracker (BoT-SORT) -> tracks that carry hand-over identities
```

*The zero-shot COCO model is scored on the same validation frames first, so the gain from fine-tuning is measured rather than assumed.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

To reproduce results without re-running training, assign `Settings.RUN_NAME` to `'INFERENCE'` and then run all cells. The notebook targets Kaggle with the GPU T4 x2 accelerator.

In [ ]:
!nvidia-smi || echo 'No NVIDIA GPU detected'

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q ultralytics pandas matplotlib

## Import Libraries

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from ultralytics import YOLO

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Settings

All paths, hyperparameters, and environment flags are centralised here. The dataset is located by searching `/kaggle/input` for the split's `data.yaml`, so the notebook does not depend on Kaggle's mount path.

In [ ]:
class Settings:
    SEED       = 42
    RUN_NAME   = 'yolov8m_8calves_v1'
    _ON_KAGGLE = Path('/kaggle/input').exists()

    INPUT_DIR  = Path('/kaggle/input') if _ON_KAGGLE else Path('../data/processed')
    OUTPUT_DIR = Path('/kaggle/working') if _ON_KAGGLE else Path('../runs/detector_8calves')

    BASE_MODEL   = 'yolov8m.pt'
    IMG_SIZE     = 800
    EPOCHS       = 120
    PATIENCE     = 30
    N_GPUS       = torch.cuda.device_count()
    BATCH_SIZE   = 8 * max(1, N_GPUS)
    _LOCAL       = 'mps' if torch.backends.mps.is_available() else 'cpu'
    DEVICE       = list(range(N_GPUS)) if N_GPUS > 1 else (0 if N_GPUS == 1 else _LOCAL)
    EVAL_DEVICE  = 0 if N_GPUS else _LOCAL
    CONF         = 0.25
    MATCH_IOU    = 0.5
    COCO_ANIMALS = list(range(15, 24))

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'GPUs: {CFG.N_GPUS}  Device: {CFG.DEVICE}  Batch: {CFG.BATCH_SIZE}  Run: {CFG.RUN_NAME}')

## Load Dataset

The split's `data.yaml` uses a relative path, so a copy with the absolute dataset root is written to the working directory for Ultralytics.

In [ ]:
source_yaml = next(CFG.INPUT_DIR.rglob('eight_calves_yolo/data.yaml'))
DATA_ROOT = source_yaml.parent
data_cfg = yaml.safe_load(source_yaml.read_text())
data_cfg['path'] = str(DATA_ROOT)
DATA_YAML = CFG.OUTPUT_DIR / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump(data_cfg))

splits = {s: sorted((DATA_ROOT / s / 'images').glob('*.png')) for s in ('train', 'val')}
print(f'Dataset root: {DATA_ROOT}')
print({s: len(v) for s, v in splits.items()})

---

# Data Check <a name="3"></a>

---

Box counts and sizes per split confirm that training and validation describe the same scene, and a few labelled frames show how crowded and occluded the calves are.

In [ ]:
def read_labels(image_path: Path) -> np.ndarray:
    txt = image_path.parent.parent / 'labels' / (image_path.stem + '.txt')
    rows = [list(map(float, l.split())) for l in txt.read_text().splitlines() if l.strip()]
    return np.array(rows).reshape(-1, 5)


rows = []
for split, images in splits.items():
    for p in images:
        labels = read_labels(p)
        rows.append({'split': split, 'boxes': len(labels),
                     'mean_w': labels[:, 3].mean() if len(labels) else np.nan,
                     'mean_h': labels[:, 4].mean() if len(labels) else np.nan})
stats = pd.DataFrame(rows)
stats.groupby('split').agg(frames=('boxes', 'size'), boxes=('boxes', 'sum'),
                           boxes_per_frame=('boxes', 'mean'), mean_w=('mean_w', 'mean'), mean_h=('mean_h', 'mean')).round(3)

In [ ]:
def draw(ax, image_path: Path, boxes_xyxy: np.ndarray, color: str, title: str) -> None:
    ax.imshow(Image.open(image_path))
    for x0, y0, x1, y1 in boxes_xyxy:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color=color, lw=1.5))
    ax.set_title(title, fontsize=9)
    ax.axis('off')


def yolo_to_xyxy(labels: np.ndarray, w: int, h: int) -> np.ndarray:
    cx, cy, bw, bh = labels[:, 1] * w, labels[:, 2] * h, labels[:, 3] * w, labels[:, 4] * h
    return np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], 1)


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, p in zip(axes, random.Random(CFG.SEED).sample(splits['val'], 3)):
    w, h = Image.open(p).size
    draw(ax, p, yolo_to_xyxy(read_labels(p), w, h), 'lime', p.name)
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'val_samples.png', dpi=110)
plt.show()

---

# Zero-Shot Baseline <a name="4"></a>

---

The COCO model has no calf class, so any of its animal classes counts as a calf detection. Predictions are matched one-to-one to labels at IoU 0.5, and the same function scores the fine-tuned model later.

In [ ]:
def iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x0 = np.maximum(a[:, None, 0], b[None, :, 0])
    y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2])
    y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def precision_recall(model: YOLO, images: list[Path], classes: list[int] | None) -> dict:
    tp = fp = fn = 0
    for p in images:
        r = model.predict(str(p), imgsz=CFG.IMG_SIZE, conf=CFG.CONF, classes=classes, verbose=False)[0]
        pred = r.boxes.xyxy.cpu().numpy()
        w, h = Image.open(p).size
        gt = yolo_to_xyxy(read_labels(p), w, h)
        ious = iou_matrix(gt, pred)
        used = set()
        for i in np.argsort(-ious.max(1)) if ious.size else []:
            j = next((j for j in np.argsort(-ious[i]) if ious[i, j] >= CFG.MATCH_IOU and j not in used), None)
            if j is not None:
                used.add(j)
        tp += len(used)
        fn += len(gt) - len(used)
        fp += len(pred) - len(used)
    return {'precision': tp / max(tp + fp, 1), 'recall': tp / max(tp + fn, 1), 'tp': tp, 'fp': fp, 'fn': fn}


results = {'zero_shot_coco': precision_recall(YOLO(CFG.BASE_MODEL), splits['val'], CFG.COCO_ANIMALS)}
results['zero_shot_coco']

---

# Fine-Tuning <a name="5"></a>

---

The detector is trained as a single class on both GPUs. Mosaic stays on because it multiplies the crowded configurations the tracker struggles with, and vertical flips are allowed because the camera looks down on the pen. Early stopping watches validation mAP.

In [ ]:
if CFG.RUN_NAME != 'INFERENCE':
    model = YOLO(CFG.BASE_MODEL)
    model.train(
        data=str(DATA_YAML), epochs=CFG.EPOCHS, patience=CFG.PATIENCE, imgsz=CFG.IMG_SIZE,
        batch=CFG.BATCH_SIZE, device=CFG.DEVICE, workers=4, single_cls=True, seed=CFG.SEED,
        cos_lr=True, fliplr=0.5, flipud=0.5, degrees=10.0, hsv_v=0.3, hsv_s=0.3, mosaic=1.0,
        close_mosaic=15, project=str(CFG.OUTPUT_DIR), name=CFG.RUN_NAME, exist_ok=True, plots=True,
    )

WEIGHTS = CFG.OUTPUT_DIR / CFG.RUN_NAME / 'weights' / 'best.pt'
print(f'Best weights: {WEIGHTS} ({WEIGHTS.stat().st_size / 1e6:.1f} MB)')

The training curves show whether validation mAP is still rising when training stops, which would mean more epochs could help.

In [ ]:
history = pd.read_csv(CFG.OUTPUT_DIR / CFG.RUN_NAME / 'results.csv')
history.columns = [c.strip() for c in history.columns]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(history['epoch'], history['train/box_loss'], label='train box loss')
axes[0].plot(history['epoch'], history['val/box_loss'], label='val box loss')
axes[0].set(title='Box loss', xlabel='Epoch')
axes[0].legend()
for col in ('metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)'):
    axes[1].plot(history['epoch'], history[col], label=col.split('/')[1])
axes[1].set(title='Validation metrics', xlabel='Epoch', ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'training_curves.png', dpi=110)
plt.show()

---

# Evaluation <a name="6"></a>

---

The best checkpoint is scored on the validation day with Ultralytics' standard metrics and with the same matching function as the zero-shot baseline, so the two rows of the comparison are directly comparable.

In [ ]:
best = YOLO(str(WEIGHTS))
val = best.val(data=str(DATA_YAML), imgsz=CFG.IMG_SIZE, single_cls=True, device=CFG.EVAL_DEVICE, plots=False,
               verbose=False)
results['fine_tuned'] = {
    **precision_recall(best, splits['val'], None),
    'map50': float(val.box.map50),
    'map50_95': float(val.box.map),
}
comparison = pd.DataFrame(results).T[['precision', 'recall', 'tp', 'fp', 'fn', 'map50', 'map50_95']]
comparison

Predictions on held-out frames, drawn next to the labels, show which situations still cause misses.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, p in enumerate(random.Random(CFG.SEED + 1).sample(splits['val'], 3)):
    w, h = Image.open(p).size
    draw(axes[0, col], p, yolo_to_xyxy(read_labels(p), w, h), 'lime', f'labels: {p.name}')
    pred = best.predict(str(p), imgsz=CFG.IMG_SIZE, conf=CFG.CONF, verbose=False)[0].boxes.xyxy.cpu().numpy()
    draw(axes[1, col], p, pred, 'red', f'fine-tuned predictions ({len(pred)})')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'val_predictions.png', dpi=110)
plt.show()

---

# Export <a name="7"></a>

---

The best weights and the comparison table are copied to the output root, where the local tracking evaluation picks them up.

In [ ]:
shutil.copy(WEIGHTS, CFG.OUTPUT_DIR / 'calf_detector_v1.pt')
(CFG.OUTPUT_DIR / 'detector_results.json').write_text(json.dumps(results, indent=1))
print(json.dumps(results, indent=1))